In [144]:
# from google.colab import files
# uploaded = files.upload()

# import os, zipfile

# os.makedirs("/root/.kaggle", exist_ok=True)
# for fn in uploaded.keys():
#     os.rename(fn, "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

# #!/bin/bash
# !kaggle competitions download -c playground-series-s6e6

# #!/bin/bash
# !kaggle datasets download fedesoriano/stellar-classification-dataset-sdss17

# # Step 4: Unzip them
# # Unzip competition
# with zipfile.ZipFile("playground-series-s6e6.zip", "r") as z:
#     z.extractall("playground-series-s6e6")

# # Unzip the simulated roads accident dataset (it will produce those CSVs)
# with zipfile.ZipFile("/content/stellar-classification-dataset-sdss17.zip", "r") as z:
#     z.extractall("stellar-classification-dataset-sdss17")

# print("ps6e6 + stellar_data")

In [145]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [146]:
train_df = pd.read_csv("playground-series-s6e6/train.csv")
test_df = pd.read_csv("playground-series-s6e6/test.csv")

In [147]:
train_df.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [148]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 577347 non-null  int64  
 1   alpha              577347 non-null  float64
 2   delta              577347 non-null  float64
 3   u                  577347 non-null  float64
 4   g                  577347 non-null  float64
 5   r                  577347 non-null  float64
 6   i                  577347 non-null  float64
 7   z                  577347 non-null  float64
 8   redshift           577347 non-null  float64
 9   spectral_type      577347 non-null  object 
 10  galaxy_population  577347 non-null  object 
 11  class              577347 non-null  object 
dtypes: float64(8), int64(1), object(3)
memory usage: 52.9+ MB


In [149]:
data_df =  pd.read_csv("/content/stellar-classification-dataset-sdss17/star_classification.csv")

In [150]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   obj_ID       100000 non-null  float64
 1   alpha        100000 non-null  float64
 2   delta        100000 non-null  float64
 3   u            100000 non-null  float64
 4   g            100000 non-null  float64
 5   r            100000 non-null  float64
 6   i            100000 non-null  float64
 7   z            100000 non-null  float64
 8   run_ID       100000 non-null  int64  
 9   rerun_ID     100000 non-null  int64  
 10  cam_col      100000 non-null  int64  
 11  field_ID     100000 non-null  int64  
 12  spec_obj_ID  100000 non-null  float64
 13  class        100000 non-null  object 
 14  redshift     100000 non-null  float64
 15  plate        100000 non-null  int64  
 16  MJD          100000 non-null  int64  
 17  fiber_ID     100000 non-null  int64  
dtypes: float64(10), int64(7),

In [151]:
def get_class_map_col(df: pd.DataFrame, col: str):
    return df[col].map({
        "STAR": 0,
        "GALAXY": 1,
        "QSO": 2
    })

In [152]:
y = get_class_map_col(train_df, "class")

In [153]:
for obj_col in train_df.select_dtypes(include="object").columns:
    print(train_df[obj_col].value_counts(), '\n\n')

spectral_type
M      303323
A/F    122122
G/K    108546
O/B     43356
Name: count, dtype: int64 


galaxy_population
Red_Sequence    319565
Blue_Cloud      257782
Name: count, dtype: int64 


class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64 




In [154]:
obj_cols = ['spectral_type', 'galaxy_population']
obj_cols

['spectral_type', 'galaxy_population']

In [155]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [156]:
df = train_df.drop(['class', 'id'], axis=1)

In [157]:
one_hot = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
transformer = ColumnTransformer(
    transformers=[
        ("one_hot", one_hot, obj_cols)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

transformer.set_output(transform="pandas")
transformed = transformer.fit_transform(df)
df1 = transformed.copy()

In [158]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 14 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   spectral_type_A/F               577347 non-null  float64
 1   spectral_type_G/K               577347 non-null  float64
 2   spectral_type_M                 577347 non-null  float64
 3   spectral_type_O/B               577347 non-null  float64
 4   galaxy_population_Blue_Cloud    577347 non-null  float64
 5   galaxy_population_Red_Sequence  577347 non-null  float64
 6   alpha                           577347 non-null  float64
 7   delta                           577347 non-null  float64
 8   u                               577347 non-null  float64
 9   g                               577347 non-null  float64
 10  r                               577347 non-null  float64
 11  i                               577347 non-null  float64
 12  z               

In [159]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

In [160]:
X = df1.copy()

In [161]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [162]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
xgb.score(X_test, y_test)

0.9662163332467307

In [163]:
balanced_accuracy_score(y_test, xgb.predict(X_test))

np.float64(0.9528885144233549)

In [164]:
test_df.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [165]:
final_test_df = transformer.transform(test_df)

In [166]:
final_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247435 entries, 0 to 247434
Data columns (total 14 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   spectral_type_A/F               247435 non-null  float64
 1   spectral_type_G/K               247435 non-null  float64
 2   spectral_type_M                 247435 non-null  float64
 3   spectral_type_O/B               247435 non-null  float64
 4   galaxy_population_Blue_Cloud    247435 non-null  float64
 5   galaxy_population_Red_Sequence  247435 non-null  float64
 6   alpha                           247435 non-null  float64
 7   delta                           247435 non-null  float64
 8   u                               247435 non-null  float64
 9   g                               247435 non-null  float64
 10  r                               247435 non-null  float64
 11  i                               247435 non-null  float64
 12  z               

In [167]:
final_test_df.columns, X_train.columns

(Index(['spectral_type_A/F', 'spectral_type_G/K', 'spectral_type_M',
        'spectral_type_O/B', 'galaxy_population_Blue_Cloud',
        'galaxy_population_Red_Sequence', 'alpha', 'delta', 'u', 'g', 'r', 'i',
        'z', 'redshift'],
       dtype='object'),
 Index(['spectral_type_A/F', 'spectral_type_G/K', 'spectral_type_M',
        'spectral_type_O/B', 'galaxy_population_Blue_Cloud',
        'galaxy_population_Red_Sequence', 'alpha', 'delta', 'u', 'g', 'r', 'i',
        'z', 'redshift'],
       dtype='object'))

In [170]:
sub_df = pd.DataFrame({
    'id': test_df['id'],
    'class': xgb.predict(final_test_df)
})

In [172]:
def get_class_map_col(df: pd.DataFrame, col: str):
    return df[col].map({
        0: "STAR",
        1: "GALAXY",
        2: "QSO"
    })

In [173]:
sub_df['class'] = get_class_map_col(sub_df, 'class')

In [175]:
sub_df.to_csv("submission.csv", index=False)